# Real-time Inference Test

Tests the deployed anomaly detection model via the ModelMesh inference endpoint.
Simulates the same request flow that the IoT consumer (`messaging`) uses.

In [ ]:
import requests
import json
import os
import urllib3
urllib3.disable_warnings()

In [ ]:
MODELMESH_URL = os.environ.get(
    'MODELMESH_URL',
    'http://modelmesh-serving.industrial-edge-tst-all:8008'
)
INFERENCE_PATH = '/v2/models/inference-service/infer'

print(f"Endpoint: {MODELMESH_URL}{INFERENCE_PATH}")

In [ ]:
def predict_vibration(value):
    """Send a vibration value to ModelMesh and return the prediction."""
    payload = {
        "inputs": [{
            "name": "predict",
            "shape": [1, 1],
            "datatype": "FP64",
            "data": [[value]]
        }]
    }
    try:
        resp = requests.post(
            f"{MODELMESH_URL}{INFERENCE_PATH}",
            json=payload,
            timeout=10,
            verify=False
        )
        if resp.status_code == 200:
            result = resp.json()
            prediction = result['outputs'][0]['data'][0]
            return 'NORMAL' if prediction == 1 else 'ANOMALY'
        else:
            return f'ERROR: {resp.status_code} {resp.text[:100]}'
    except requests.exceptions.ConnectionError:
        return 'ERROR: Cannot connect to ModelMesh (check namespace/service)'
    except Exception as e:
        return f'ERROR: {str(e)}'

## Test with known values

In [ ]:
test_cases = [
    (12.0, 'Normal vibration'),
    (15.0, 'Normal vibration'),
    (17.5, 'High-normal vibration'),
    (25.0, 'Above normal threshold'),
    (35.0, 'Anomaly spike'),
    (45.0, 'Severe anomaly'),
]

print(f"{'Value':>8}  {'Expected':>20}  {'Prediction':>12}")
print('-' * 48)
for value, desc in test_cases:
    result = predict_vibration(value)
    print(f"{value:8.1f}  {desc:>20}  {result:>12}")

## Simulate streaming sensor data

In [ ]:
import numpy as np
import time

np.random.seed(42)
n_readings = 20

readings = np.concatenate([
    np.random.uniform(10, 18, 15),   # 15 normal
    np.random.uniform(30, 50, 5),    # 5 anomalies
])
np.random.shuffle(readings)

anomaly_count = 0
for i, val in enumerate(readings):
    result = predict_vibration(val)
    if result == 'ANOMALY':
        anomaly_count += 1
    marker = ' ⚠' if result == 'ANOMALY' else ''
    print(f"[{i+1:2d}/{n_readings}] Vibration={val:6.2f}  →  {result}{marker}")

print(f"\nSummary: {anomaly_count} anomalies detected in {n_readings} readings")